# Vietnamese keyframe OCR — Paddle detector + VietOCR (v3.1 — batch32 + long-line segmentation)
        **Python3.13 / Kaggle Linux / T4 GPU0 / Internet required.** New isolated runtime and new output names; original notebook/data untouched.

        Fixes the old Latin recognizer's missing Vietnamese alphabet. Uses pinned VietOCR VGG-Transformer inference source with full Vietnamese vocabulary, strict tensor-only checkpoint load, and native-image detection/crops. A complete alphabet is necessary, **not a guarantee of accuracy** on blurred video.

        Start with preview. Inspect boxes, raw lines/confidence and omitted/truncated lines before setting `RUN_FULL=True`. No captions, translations, guessed text correction, or generative upscaling. GPU/API/model/alphabet failures stop; no CPU/old-Latin fallback.

        **VietOCR now uses real tensor batches of up to32 crops.** Full OCR buffers up to32 uncached frames to fill exact-resized-width buckets, without extra image padding or mixed precision. Detector remains one image at a time on GPU0. Width buckets/final tails can be smaller than32. Inspect actual batch histogram in logs/run.json; no speedup or VRAM claim until measured on Kaggle. OOM stops instead of silently shrinking batches. Old v3 scalar ZIPs/checkpoints are preserved.
        ## 1. Input and quality controls

In [ ]:
INPUT_ROOT = '/kaggle/input/your-keyframes/L01'
RUN_FULL = False  # First Run All previews only. Set True after checking preview.
PREVIEW_COUNT = 8  # Evenly spaced real frames in this subtree, at most50.
PREVIEW_IDENTITIES = []  # Optional exact list, e.g. ['L01_V005:69', 'L01_V013:170'].
PUBLIC_MIN_CONFIDENCE = 0.5  # Uncalibrated; raw lines always kept in diagnostics.
RUN_OK = False
EXPORT_OK = False


## 2. Isolated runtime
        No installs into the notebook kernel. TWO owned pip-less venvs/processes keep Paddle3.2 and Torch2.8 NVIDIA dependencies separate. Each gets its own resolver, core pins and mandatory pip check. Old mixed venv is preserved and not reused. No package-version fallback or compilation. Python starts with -I -S and an explicit venv-only launcher, avoiding Kaggle global sitecustomize/wrapt hooks without suppressing dependency checks.

        VietOCR PyPI's old Pillow pin is incompatible with Python3.13. Instead this worker downloads only an explicit inference-source allowlist from `pbcquoc/vietocr` commit`fe8c3a7...`, checks Git blob hashes and retains its license. No training/augmentation stack, `pip install vietocr --no-deps`, or gdown workaround. Model weights and detector assets are pinned and hash-checked; only the vgg checkpoint mirror at`doanhm/vgg_transformer` is used, with its provenance recorded.

        **Not GPU-validated on Kaggle yet.** Resolver, strict state load, both GPU smokes and alphabet gate must pass there before bulk runs. Downloading model weights happens only when preview/run is explicitly executed in this notebook. Expect the first run to take several minutes.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys
import uuid

RUN_OK = False
EXPORT_OK = False
WORKER_SOURCE = r'''
import argparse
import hashlib
import importlib.metadata
import io
import json
import math
import os
from pathlib import Path
import re
import shutil
import stat
import sys
import traceback
import uuid
import zipfile

SCHEMA = 'vietnamese-kaggle-ocr-v1'
PINS = {'numpy': '2.2.6', 'Pillow': '11.3.0', 'tqdm': '4.67.1', 'opencv-contrib-python': '4.10.0.84', 'torch': '2.8.0+cu126', 'torchvision': '0.23.0+cu126', 'einops': '0.8.1', 'PyYAML': '6.0.2'}
MODELS = {'detection': 'PP-OCRv5_server_det',
          'recognition': 'latin_PP-OCRv5_mobile_rec'}
OPTIONS = dict(device='gpu:0', enable_hpi=False, use_tensorrt=False,
               text_detection_model_name=MODELS['detection'],
               text_recognition_model_name=MODELS['recognition'],
               use_doc_orientation_classify=False, use_doc_unwarping=False,
               use_textline_orientation=False, text_det_limit_side_len=960,
               text_det_limit_type='max', text_det_thresh=0.3,
               text_det_box_thresh=0.6, text_det_unclip_ratio=1.5,
               text_rec_score_thresh=0.0, text_recognition_batch_size=1)


def digest(raw):
    return hashlib.sha256(raw).hexdigest()


def encoded(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(',', ':'), allow_nan=False).encode('utf-8')


def safe_path(value):
    path = Path(value)
    if not path.is_absolute() or '..' in path.parts:
        raise ValueError('Use absolute paths without traversal')
    for part in [path, *path.parents]:
        try:
            info = part.lstat()
        except FileNotFoundError:
            continue
        if stat.S_ISLNK(info.st_mode) or getattr(info, 'st_file_attributes', 0) & 0x400:
            raise ValueError(f'Symlink/reparse path rejected: {part}')
    return path.resolve()


def safe_tree(root):
    root = safe_path(root)
    if root.exists():
        for folder, dirs, files in os.walk(root, followlinks=False):
            if len(Path(folder).relative_to(root).parts) > 8:
                raise ValueError('Directory nesting exceeds 8 levels')
            for name in dirs + files:
                safe_path(Path(folder) / name)
    return root


def safe_runtime_tree(root):
    # venv lib64 and downloaded model caches may have generated INTERNAL links.
    # Never allow an existing runtime link to redirect pip/cache writes outside it.
    root = safe_path(root)
    for folder, dirs, files in os.walk(root, followlinks=False):
        for name in dirs + files:
            path = Path(folder) / name
            info = path.lstat()
            if stat.S_ISLNK(info.st_mode) or getattr(info, 'st_file_attributes', 0) & 0x400:
                target = path.resolve()
                if target == root or root not in target.parents:
                    raise ValueError(f'External runtime symlink/reparse path rejected: {path}')


def validate_paths(input_root, output, env_root):
    paths = [safe_path(p) for p in (input_root, output, env_root)]
    for i, first in enumerate(paths):
        for second in paths[i + 1:]:
            if first == second or first in second.parents or second in first.parents:
                raise ValueError('INPUT_ROOT, output and venv must be disjoint in BOTH directions')
    safe_tree(paths[0])
    safe_tree(paths[1])
    safe_runtime_tree(paths[2])
    if not paths[0].is_dir():
        raise ValueError('INPUT_ROOT must be an existing directory')
    return paths


COPY_SUFFIX = re.compile(r'\s*\(\d+\)\s*$')


def inventory(root):
    root = safe_tree(root)
    items, seen, export_paths = [], set(), set()
    for path in sorted(root.rglob('*')):
        if not path.is_file() or path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}:
            continue
        # Skip copy-style names such as '001 (1).jpg' (also inside folders like 'L04_V001 (1)').
        if COPY_SUFFIX.search(path.stem) or any(COPY_SUFFIX.search(p) for p in path.relative_to(root).parts[:-1]):
            continue
        match = re.fullmatch(r'L([0-9]{1,4})_V([0-9]{1,6})', path.parent.name, re.I)
        if not match or not re.fullmatch(r'[0-9]{1,12}', path.stem):
            raise ValueError(f'Expected video folder / numeric keyframe filename: {path}')
        video = f'L{int(match[1]):02d}_V{int(match[2]):03d}'
        if path.parent.name != video:
            raise ValueError(f'Expected canonical video folder {video}: {path}')
        number = int(path.stem)
        identity = f'{video}:{number}'
        group = video.split('_V')[0]
        export_path = f'{group}/{video}/{path.stem}.json'
        if identity in seen:
            raise ValueError(f'Duplicate canonical identity: {identity}')
        if export_path in export_paths:
            raise ValueError(f'Duplicate export path: {export_path}')
        seen.add(identity)
        export_paths.add(export_path)
        items.append(dict(id=identity, video_id=video, keyframe_n=number,
                          relative_path=path.relative_to(root).as_posix()))
    if not items:
        raise ValueError('No keyframe images inside INPUT_ROOT')
    return sorted(items, key=lambda x: (x['video_id'], x['keyframe_n']))


def atomic_bytes(path, raw):
    path = safe_path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    created = False
    try:
        with temporary.open('xb') as stream:
            created = True
            stream.write(raw)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if created:
            temporary.unlink(missing_ok=True)


def atomic_json(path, value):
    atomic_bytes(path, encoded(value) + b'\n')


def read_json(path):
    return json.loads(safe_path(path).read_text(encoding='utf-8'))


def claim_output(root, output, create=True):
    owner = {'schema': SCHEMA, 'input_root': str(root)}
    marker = output / 'owner.json'
    if output.exists() and any(output.iterdir()):
        if not marker.is_file() or read_json(marker) != owner:
            raise ValueError('Refusing output owned by another dataset or unrelated files')
    elif create:
        atomic_json(marker, owner)


def runtime_identity(env_root):
    marker = env_root / 'simple-runtime.json'
    expected = {'schema': 'vietnamese-torch-runtime-v2', 'pins': PINS}
    if env_root.exists() and any(env_root.iterdir()):
        if not marker.is_file() or read_json(marker) != expected:
            raise RuntimeError('Unowned/incompatible venv; refusing to modify or delete it')
    return marker, expected


class PublicationCollision(ValueError):
    pass


def validate_publish(root, output, env_root, archive):
    root, output, env_root = validate_paths(root, output, env_root)
    archive = safe_path(archive)
    for path in (root, output, env_root):
        if (path == archive or path in archive.parents or archive in path.parents
                or archive.parent == path or archive.parent in path.parents):
            raise ValueError('Publish directory and input/scratch must be disjoint in BOTH directions')
    if archive.exists():
        receipt = output / 'publication.json'
        if not receipt.is_file():
            raise PublicationCollision('Unrelated preexisting ocr.zip; refusing overwrite')
        owned = read_json(receipt)
        if (owned.get('archive') != str(archive) or owned.get('input_root') != str(root)
                or owned.get('archive_sha256') != file_digest(archive)):
            raise PublicationCollision('Unrelated or changed preexisting ocr.zip; refusing overwrite')
    return archive


def file_digest(path):
    value = hashlib.sha256()
    with safe_path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def writable_probe(folder):
    folder = safe_path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    probe = folder / ('.write-probe-' + uuid.uuid4().hex)
    created = False
    try:
        with probe.open('xb') as stream:
            created = True
            stream.write(b'probe')
            stream.flush()
            os.fsync(stream.fileno())
    finally:
        if created:
            probe.unlink(missing_ok=True)


def select_scratch(root, archive, candidates=('/kaggle/temp', '/tmp')):
    root = safe_tree(root)
    key = digest(str(root).encode())
    owner = dict(schema='vietnamese-kaggle-lines-batch32-scratch-v3', input_root=str(root))
    failures, collisions = [], []
    for candidate in candidates:
        scratch = safe_path(Path(candidate) / ('ocr-vietnamese-lines-v3-batch32-' + key))
        if root == scratch or root in scratch.parents or scratch in root.parents:
            raise ValueError('Scratch and input must be disjoint in BOTH directions')
        output, runtime = scratch / 'state', scratch / 'runtime'
        # Validate every destination and ownership BEFORE even a permission probe.
        try:
            validate_publish(root, output, runtime, archive)
        except PublicationCollision as exc:
            # A previous fallback profile may own the ZIP; never write here.
            collisions.append(str(exc))
            continue
        safe_runtime_tree(scratch)
        marker = scratch / 'scratch-owner.json'
        if scratch.exists() and any(scratch.iterdir()):
            if not marker.is_file() or read_json(marker) != owner:
                raise ValueError('Unowned scratch; refusing to modify or delete it')
        claim_output(root, output, create=False)
        runtime_identity(runtime)
        try:
            writable_probe(scratch)
        except OSError as exc:
            failures.append(f'{scratch}: {exc}')
            continue
        if not marker.exists():
            atomic_json(marker, owner)
        claim_output(root, output)
        return scratch, output, runtime
    if collisions:
        raise PublicationCollision('Unrelated preexisting ocr.zip or unavailable owning scratch; refusing overwrite')
    raise RuntimeError('No writable transient scratch (never using working): ' + '; '.join(failures))


def isolated_environment(scratch, inherited):
    scratch = safe_path(scratch)
    # Inherit capabilities, not arbitrary cache/log/profiling write destinations.
    allowed = {'PATH', 'SYSTEMROOT', 'SystemRoot', 'WINDIR', 'COMSPEC', 'PATHEXT',
               'LANG', 'LC_ALL', 'LC_CTYPE', 'TZ', 'TERM', 'NO_COLOR',
               'LD_LIBRARY_PATH', 'CUDA_HOME', 'CUDA_PATH', 'CUDA_MODULE_LOADING',
               'CUDA_DEVICE_MAX_CONNECTIONS', 'CUDA_LAUNCH_BLOCKING',
               'NVIDIA_VISIBLE_DEVICES', 'NVIDIA_DRIVER_CAPABILITIES',
               'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
               'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY',
               'http_proxy', 'https_proxy', 'all_proxy', 'no_proxy',
               'SSL_CERT_FILE', 'SSL_CERT_DIR', 'REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE',
               'HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'MODELSCOPE_API_TOKEN'}
    environment = {k: v for k, v in inherited.items() if k in allowed}
    routes = {'HOME': 'home', 'USERPROFILE': 'home', 'XDG_CACHE_HOME': 'cache',
              'XDG_CONFIG_HOME': 'config', 'XDG_DATA_HOME': 'data', 'XDG_STATE_HOME': 'local-state',
              'PIP_CACHE_DIR': 'cache/pip', 'HF_HOME': 'cache/huggingface',
              'HF_HUB_CACHE': 'cache/huggingface/hub',
              'HUGGINGFACE_HUB_CACHE': 'cache/huggingface/hub',
              'HF_ASSETS_CACHE': 'cache/huggingface/assets',
              'HUGGINGFACE_ASSETS_CACHE': 'cache/huggingface/assets',
              'TRANSFORMERS_CACHE': 'cache/huggingface/hub',
              'PYTORCH_TRANSFORMERS_CACHE': 'cache/huggingface/hub',
              'PYTORCH_PRETRAINED_BERT_CACHE': 'cache/huggingface/hub',
              'PADDLE_HOME': 'cache/paddle', 'PADDLE_OCR_BASE_DIR': 'cache/paddleocr',
              'PADDLE_PDX_CACHE_HOME': 'cache/paddlex', 'PADDLEX_HOME': 'cache/paddlex',
              'MODELSCOPE_CACHE': 'cache/modelscope', 'TORCH_HOME': 'cache/torch',
              'CUDA_CACHE_PATH': 'cache/cuda', 'MPLCONFIGDIR': 'config/matplotlib',
              'TMPDIR': 'tmp', 'TEMP': 'tmp', 'TMP': 'tmp'}
    for key, relative in routes.items():
        path = safe_path(scratch / relative)
        path.mkdir(parents=True, exist_ok=True)
        environment[key] = str(path)
    environment.update(CUDA_VISIBLE_DEVICES='0', CUDA_DEVICE_ORDER='PCI_BUS_ID',
                       PIP_CONFIG_FILE=os.devnull, PYTHONNOUSERSITE='1', PYTHONUNBUFFERED='1',
                       GLOG_logtostderr='1')
    return environment


def invalidate_ready(output):
    safe_path(output / 'ready.json').unlink(missing_ok=True)


def invalidate_current_ready(root, output, env_root, run_id):
    root, output, env_root = validate_paths(root, output, env_root)
    claim_output(root, output, create=False)
    path = safe_path(output / 'ready.json')
    try:
        ready = read_json(path)
    except FileNotFoundError:
        return
    if (isinstance(ready, dict) and ready.get('schema') == SCHEMA
            and ready.get('input_root') == str(root) and ready.get('run_id') == run_id):
        path.unlink()  # Never remove publication ownership or another invocation's receipt.


def checked_ready(root, output, env_root, run_id, archive):
    archive = validate_publish(root, output, env_root, archive)
    ready = read_json(output / 'ready.json')
    state = read_json(output / 'run.json')
    if (ready.get('run_id') != run_id or state.get('run_id') != run_id
            or state.get('phase') != 'finished'):
        raise RuntimeError('Export invocation mismatch')
    if (ready.get('profile_sha256') != state.get('profile_sha256')
            or digest(encoded(read_json(output / 'profile.json'))) != ready.get('profile_sha256')
            or ready != read_json(output / 'publication.json')
            or ready.get('archive_sha256') != file_digest(archive)):
        raise RuntimeError('Export receipt/digest mismatch')
    return archive


def publish_zip(root, output, env_root, run_id, source, archive, summary):
    archive = validate_publish(root, output, env_root, archive)
    archive.parent.mkdir(parents=True, exist_ok=True)
    temporary = archive.parent / ('.ocr-publish-' + uuid.uuid4().hex + '.tmp')
    created = False
    try:
        # Copy, not rename: scratch and working may be on different filesystems.
        with temporary.open('xb') as target:
            created = True
            with source.open('rb') as stream:
                shutil.copyfileobj(stream, target)
            target.flush()
            os.fsync(target.fileno())
        if file_digest(temporary) != file_digest(source):
            raise RuntimeError('Publication copy integrity failure')
        with zipfile.ZipFile(temporary) as check:
            if check.testzip() is not None:
                raise RuntimeError('Invalid publication ZIP')
        receipt = dict(summary, archive=str(archive), archive_sha256=file_digest(temporary))
        os.replace(temporary, archive)
        atomic_json(output / 'publication.json', receipt)
        atomic_json(output / 'ready.json', receipt)  # Scratch only; written last.
    finally:
        if created:
            temporary.unlink(missing_ok=True)  # Only this invocation's own file.
    return archive


def decode(raw):
    from PIL import Image, ImageOps
    # Deliberately no exif_transpose: coordinates refer to the stored raster.
    with Image.open(io.BytesIO(raw)) as source:
        source.load()  # Force lazy decoder exceptions into the decode-only handler.
        image = source.convert('RGB')
    return ImageOps.expand(image, border=32, fill='white'), image.size


def decode_or_error(raw):
    from PIL import Image
    try:
        return decode(raw), None
    except (OSError, SyntaxError, ValueError, Image.DecompressionBombError) as exc:
        return None, {'type': type(exc).__name__, 'message': str(exc)}


def require_gpu(paddle):
    if os.environ.get('CUDA_VISIBLE_DEVICES') != '0':
        raise RuntimeError('CUDA_VISIBLE_DEVICES must be exactly 0 before importing paddle')
    if not paddle.is_compiled_with_cuda() or paddle.device.cuda.device_count() != 1:
        raise RuntimeError('A CUDA build and exactly one visible GPU are required; no CPU fallback')
    name = paddle.device.cuda.get_device_name(0)
    if 'T4' not in name:
        raise RuntimeError(f'Expected selected NVIDIA T4, got {name}')
    paddle.set_device('gpu:0')
    value = paddle.to_tensor([[2.0]], place=paddle.CUDAPlace(0))
    result = paddle.matmul(value, value)
    paddle.device.cuda.synchronize()
    if not result.place.is_gpu_place() or float(result.numpy()[0][0]) != 4.0:
        raise RuntimeError('Actual CUDA tensor operation failed')
    print(f'CUDA_VISIBLE_DEVICES=0; physical GPU 0 -> gpu:0: {name}; CUDA tensor smoke OK', flush=True)
    return {'device': 'gpu:0', 'physical_gpu': 0, 'gpu_name': name,
            'cuda': paddle.version.cuda(), 'cudnn': paddle.version.cudnn()}


class Adapter:
    def __init__(self):
        os.environ['CUDA_VISIBLE_DEVICES'] = '0'
        import paddle
        gpu = require_gpu(paddle)
        versions = {name: importlib.metadata.version(name) for name in PINS}
        if versions != PINS:
            raise RuntimeError(f'Core version mismatch: {versions}; expected {PINS}')
        from paddleocr import PaddleOCR
        self.ocr = PaddleOCR(**OPTIONS)
        pipe = self.ocr.paddlex_pipeline
        models = {'detection': pipe.text_det_model.model_name,
                  'recognition': pipe.text_rec_model.model_name}
        if models != MODELS:
            raise RuntimeError(f'Unexpected actual models: {models}')
        for model in (pipe.text_det_model, pipe.text_rec_model):
            if model.pp_option.device_type != 'gpu' or model.pp_option.device_id != 0:
                raise RuntimeError('OCR predictor is not on gpu:0')
        self.engine = dict(gpu, models=models, core_versions=versions,
                           packages=dict(sorted((d.metadata['Name'], d.version)
                                                for d in importlib.metadata.distributions())))
        print(json.dumps(self.engine, ensure_ascii=False, indent=2), flush=True)

    def recognize(self, padded):
        import numpy as np
        # PaddleX's ndarray reader expects BGR, not Pillow RGB.
        results = self.ocr.predict(input=np.asarray(padded)[:, :, ::-1].copy())
        if len(results) != 1 or 'error' in results[0]:
            raise RuntimeError(f'Unexpected OCR result: {results}')
        result = results[0]
        texts, scores, polygons = (result[k] for k in ('rec_texts', 'rec_scores', 'rec_polys'))
        if not len(texts) == len(scores) == len(polygons):
            raise RuntimeError('OCR API returned inconsistent line arrays')
        return [dict(text=text, confidence=float(score), polygon_padded=poly.tolist())
                for text, score, poly in zip(texts, scores, polygons)]


def make_profile(adapter):
    profile = dict(schema=SCHEMA, engine=adapter.engine, options=OPTIONS,
                   padding_px=32, wrapper_resize=False, exif_rotation=False,
                   worker_sha256=digest(Path(__file__).read_bytes()))
    return profile, digest(encoded(profile))


def record_for(item, sha, profile_hash, decoded, error, adapter):
    record = dict(item, schema=SCHEMA, source_sha256=sha, profile_sha256=profile_hash,
                  mapping=None, status='error' if error else 'empty', error=error, lines=[],
                  geometry=None)
    if error:
        return record
    padded, (width, height) = decoded
    record['geometry'] = dict(original_size=[width, height], padded_size=list(padded.size),
                              padding_ltrb=[32, 32, 32, 32], wrapper_resize=False,
                              original_to_padded=[[1, 0, 32], [0, 1, 32], [0, 0, 1]],
                              padded_to_original=[[1, 0, -32], [0, 1, -32], [0, 0, 1]])
    for line in adapter.recognize(padded):
        polygon = [[float(x), float(y)] for x, y in line['polygon_padded']]
        score = float(line['confidence'])
        if len(polygon) != 4 or not isinstance(line['text'], str) or not 0 <= score <= 1:
            raise RuntimeError('Invalid OCR line API values')
        if not all(math.isfinite(v) for point in polygon for v in point):
            raise RuntimeError('Non-finite polygon')
        original = [[x - 32, y - 32] for x, y in polygon]
        clipped = [[max(0, min(width, x)), max(0, min(height, y))] for x, y in original]
        record['lines'].append(dict(text=line['text'], confidence=score,
                                    polygon_padded=polygon, polygon_original_unclipped=original,
                                    polygon_original_clipped=clipped))
    record['status'] = 'ok' if record['lines'] else 'empty'
    return record


def checkpoint_path(output, item):
    return output / 'checkpoints' / (digest(item['id'].encode()) + '.json')


def save_checkpoint(output, record):
    atomic_json(checkpoint_path(output, record),
                {'record': record, 'record_sha256': digest(encoded(record))})


def cached(output, item, sha, profile_hash):
    path = checkpoint_path(output, item)
    if not path.exists():
        return None
    try:
        envelope = read_json(path)
        record = envelope['record']
        if envelope['record_sha256'] != digest(encoded(record)):
            return None
        expected = dict(item, schema=SCHEMA, source_sha256=sha, profile_sha256=profile_hash, mapping=None)
        if any(record[k] != value for k, value in expected.items()):
            return None
        if record['status'] not in {'ok', 'empty', 'error'} or not isinstance(record['lines'], list):
            return None
        if record['status'] in {'ok', 'empty'}:
            if record['error'] is not None or not record['geometry']:
                return None
            if (record['status'] == 'ok') != bool(record['lines']):
                return None
        if record['status'] == 'error' and (not record['error'] or record['lines']):
            return None
        return record
    except (ValueError, KeyError, TypeError):
        print(f'Invalid checkpoint, recompute: {path.name}', flush=True)
        return None


def run(root, output, env_root, run_id, adapter=None, progress=None):
    root, output, env_root = validate_paths(root, output, env_root)
    claim_output(root, output)
    state = {'run_id': run_id, 'phase': 'running', 'complete': False}
    invalidate_ready(output)
    atomic_json(output / 'run.json', state)
    try:
        items = inventory(root)  # Reject duplicates before model load or bulk work.
        adapter = adapter if adapter is not None else Adapter()
        profile, profile_hash = make_profile(adapter)
        # A real decodable input, the SAME model/adapter/padding path as bulk.
        smoke = None
        for item in items:
            raw = safe_path(root / item['relative_path']).read_bytes()
            decoded, error = decode_or_error(raw)
            if not error:
                smoke = record_for(item, digest(raw), profile_hash, decoded, None, adapter)
                break
        if smoke is None:
            raise RuntimeError('No decodable real image for mandatory GPU OCR smoke')
        print(f'Real image OCR smoke OK: {smoke["relative_path"]}; profile={profile_hash}', flush=True)
        atomic_json(output / 'profile.json', profile)
        if progress is None:
            from tqdm import tqdm
            progress = lambda rows: tqdm(rows, total=len(rows), unit='image',
                                         desc='OCR (including resume checks)', file=sys.stdout,
                                         mininterval=1, smoothing=0.1)
        errors, skipped = 0, 0
        for item in progress(items):
            raw = safe_path(root / item['relative_path']).read_bytes()
            sha = digest(raw)
            old = cached(output, item, sha, profile_hash)
            if old and old['status'] in {'ok', 'empty'}:
                skipped += 1
                continue
            if item['id'] == smoke['id'] and sha == smoke['source_sha256']:
                record = smoke
            else:
                decoded, error = decode_or_error(raw)
                # Backend exceptions are intentionally OUTSIDE the decode handler.
                record = record_for(item, sha, profile_hash, decoded, error, adapter)
            save_checkpoint(output, record)  # fsync + atomic replace after every image.
            if record['status'] == 'error':
                errors += 1
                print(f'Image decode error: {item["relative_path"]}: {record["error"]}', flush=True)
        state.update(phase='finished', complete=errors == 0, errors=errors,
                     skipped=skipped, total=len(items), profile_sha256=profile_hash, inventory=items)
        atomic_json(output / 'run.json', state)
        print(f'Finished: {len(items)} images; resumed {skipped}; decode errors {errors}', flush=True)
    except BaseException as exc:
        state.update(phase='fatal', complete=False, error=f'{type(exc).__name__}: {exc}')
        atomic_json(output / 'run.json', state)
        traceback.print_exc()
        raise


def public_record(record):
    # Project only integrity-verified checkpoints; preserve model text and order.
    text = '' if record['status'] == 'error' else '\n'.join(
        line['text'] for line in record['lines'] if line['text'].strip())
    video = record['video_id']
    group = video.split('_V')[0]
    return dict(video_id=video, frame_id=None, keyframe_n=record['keyframe_n'],
                image_path=f'data/{group}/{video}/{Path(record["relative_path"]).name}',
                ocr_text=text, retrieval_text=text,
                status='error' if record['status'] == 'error' else ('ok' if text else 'empty'))


def export_current(root, output, env_root, run_id, archive='/kaggle/working/ocr.zip'):
    root, output, env_root = validate_paths(root, output, env_root)
    archive = validate_publish(root, output, env_root, archive)
    claim_output(root, output)
    invalidate_ready(output)
    state = read_json(output / 'run.json')
    if state['run_id'] != run_id or state['phase'] != 'finished':
        raise RuntimeError('No successful current invocation; refusing stale downloads')
    profile = read_json(output / 'profile.json')
    if digest(encoded(profile)) != state['profile_sha256']:
        raise RuntimeError('Profile integrity failure')
    items = inventory(root)
    if items != state['inventory']:
        raise RuntimeError('Input inventory changed since run; run cell 3 again')
    records, missing = [], []
    for item in items:
        sha = digest(safe_path(root / item['relative_path']).read_bytes())
        record = cached(output, item, sha, state['profile_sha256'])
        if record is None:
            missing.append(item['id'])
        else:
            records.append(record)
    if len({r['id'] for r in records}) != len(records):
        raise RuntimeError('Duplicate final IDs')
    public_records = [public_record(record) for record in records]
    errors = sum(r['status'] == 'error' for r in public_records)
    summary = dict(schema=SCHEMA, run_id=run_id, input_root=str(root),
                   total=len(items), exported=len(records), errors=errors, missing=missing,
                   ok=sum(r['status'] == 'ok' for r in public_records),
                   empty=sum(r['status'] == 'empty' for r in public_records),
                   complete=not errors and not missing, profile_sha256=state['profile_sha256'])
    # Fresh invocation directory: never serve old per-image JSON/ZIP after a failure.
    destination = output / 'downloads' / run_id
    if destination.exists():
        raise RuntimeError('Export already exists; rerun cell 3 for a fresh export')
    names = []
    for record, public in zip(records, public_records):
        # Inventory validated canonical folders, numeric stems and collisions before writes.
        video = record['video_id']
        group = video.split('_V')[0]  # Derive from identity, never input ancestors.
        name = f'{group}/{video}/{Path(record["relative_path"]).stem}.json'
        atomic_bytes(destination / name, (json.dumps([public], ensure_ascii=False, indent=2,
                                                    allow_nan=False) + '\n').encode('utf-8'))
        names.append(name)
    temporary = destination / 'ocr.zip.tmp'
    with zipfile.ZipFile(temporary, 'x', compression=zipfile.ZIP_DEFLATED) as packed:
        # Explicit allowlist: ONLY per-image OCR records, nothing else.
        for name in names:
            packed.write(destination / name, name)
    with zipfile.ZipFile(temporary) as check:
        if check.namelist() != names or check.testzip() is not None:
            raise RuntimeError('Export ZIP integrity failure')
    os.replace(temporary, destination / 'ocr.zip')
    try:
        publish_zip(root, output, env_root, run_id, destination / 'ocr.zip', archive, summary)
        print(json.dumps(summary, indent=2), flush=True)  # Log only; not in the ZIP.
        print(f'Download: {archive}', flush=True)
        return destination
    except BaseException:
        invalidate_current_ready(root, output, env_root, run_id)
        raise


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('action', choices=['run', 'export'])
    parser.add_argument('input_root')
    parser.add_argument('output')
    parser.add_argument('env_root')
    parser.add_argument('run_id')
    parser.add_argument('--archive', default='/kaggle/working/ocr.zip')
    args = parser.parse_args()
    if not re.fullmatch(r'[a-f0-9]{32}', args.run_id):
        raise ValueError('Invalid invocation ID')
    validate_publish(args.input_root, args.output, args.env_root, args.archive)
    if args.action == 'run':
        run(args.input_root, args.output, args.env_root, args.run_id)
    else:
        export_current(args.input_root, args.output, args.env_root, args.run_id, args.archive)




PADDLE_PINS = {'paddlepaddle-gpu': '3.2.2', 'paddleocr': '3.2.0', 'paddlex': '3.2.0', 'numpy': '2.2.6', 'Pillow': '11.3.0', 'tqdm': '4.67.1', 'opencv-contrib-python': '4.10.0.84'}

"""Embedded into the standalone Vietnamese notebook AFTER legacy IO helpers.

This file is a notebook source fragment, not a standalone module. Heavy imports
are lazy. Existing datasets, notebook, venv and published OCR remain untouched.
"""

VI_REVISION = 'fe8c3a7fc714aec57ab81cec844eb3adf0c1636c'
VI_BLOBS = {
    'LICENSE': '261eeb9e9f8b2b4b0d119366dda99c6fd7d35c64',
    'config/base.yml': '08a9fca6233a19fa1da42aed6214958e4284ea97',
    'config/vgg-transformer.yml': 'ca7dbbca06177d02e472821b114a751554833294',
    'vietocr/__init__.py': 'e69de29bb2d1d6434b8b29ae775ad8c2e48c5391',
    'vietocr/model/__init__.py': 'e69de29bb2d1d6434b8b29ae775ad8c2e48c5391',
    'vietocr/model/backbone/__init__.py': 'e69de29bb2d1d6434b8b29ae775ad8c2e48c5391',
    'vietocr/model/backbone/cnn.py': '1af65da7bf0c6105e0cefe36302e67f05745b88a',
    'vietocr/model/backbone/vgg.py': 'b164b9c783c2e4106b2e0701d16f0cc181924970',
    'vietocr/model/backbone/resnet.py': 'b2aff464b44015f5941e8711e12b8a585026b786',
    'vietocr/model/seqmodel/__init__.py': 'e69de29bb2d1d6434b8b29ae775ad8c2e48c5391',
    'vietocr/model/seqmodel/transformer.py': 'a7d4ed5313cf4031eb60012677cdbe2a8553eba2',
    'vietocr/model/seqmodel/seq2seq.py': '31960ed313d6dccf3635fc45371e12d8ef448032',
    'vietocr/model/seqmodel/convseq2seq.py': '46b980b3033ec13b1b17d457df037de183162636',
    'vietocr/model/transformerocr.py': '7c88cd9860592c3a699652397a9ba4ec60bd2926',
    'vietocr/model/vocab.py': 'a4fab4059ad3098e57388d83ea49be290e6d7b9d',
}
WEIGHTS_URL = ('https://huggingface.co/doanhm/vgg_transformer/resolve/'
               '4de5b0429954fdac3dee3173978c062b3bc4d534/vgg_transformer.pth')
WEIGHTS_SHA256 = '380512193a8b6cbf6fad80deacdc9b6939d10d473d199892fc6408d13775ea59'
DET_BASE = ('https://huggingface.co/PaddlePaddle/PP-OCRv5_server_det/resolve/'
            'ca867c897ecbca8873081573a802ad70d499cb94/')
DET_ASSETS = {
    'inference.json': ('git', '7b582d63397b09eea2b3ee8c7c7e4efae5d0ccff'),
    'inference.yml': ('git', 'c4f16313e7e2c567da8e9fa8f4aeba14a307786e'),
    'inference.pdiparams': ('sha256', '183146fe9d9910352f68482f623bcbbb9fa7b9e8fa1463b9ad288cef00524d2d'),
}
VI_LOWER = ('aàáảãạăằắẳẵặâầấẩẫậeèéẻẽẹêềếểễệiìíỉĩị'
            'oòóỏõọôồốổỗộơờớởỡợuùúủũụưừứửữựyỳýỷỹỵđ')
VI_REQUIRED = VI_LOWER + VI_LOWER.upper()
PUBLIC_MIN_CONFIDENCE = float(os.environ.get('KIS_VI_OCR_MIN_CONFIDENCE', '0.5'))
if not math.isfinite(PUBLIC_MIN_CONFIDENCE) or not 0 <= PUBLIC_MIN_CONFIDENCE <= 1:
    raise ValueError('Public confidence threshold must be in [0,1]')
OPTIONS = dict(detector_limit_side_len=1600, detector_limit_type='max',
               detector_thresh=0.3, detector_box_thresh=0.6, detector_unclip_ratio=1.5,
               recognition_image_height=32, recognition_max_width=2048,
               recognition_max_tokens=256, recognition_batch_size=32,
               recognition_frame_window=32, recognition_batch_policy='exact_width_no_padding_fp32_v1',
               line_segment_trigger_width=512, line_segment_max_width=480,
               line_segment_min_gap_height_ratio=0.18, line_segment_max_count=32,
               line_segment_method='otsu_word_gaps_no_overlap_v1',
               public_min_confidence=PUBLIC_MIN_CONFIDENCE, use_super_resolution=False)
MODELS = dict(detection='PP-OCRv5_server_det', recognition='VietOCR/vgg_transformer')


def git_blob_digest(raw):
    return hashlib.sha1(b'blob ' + str(len(raw)).encode() + b'\0' + raw).hexdigest()


def fetch_asset(url, path, kind, expected, max_bytes):
    """Known HTTPS assets only, verify cached bytes too; no unpickling here."""
    from urllib.request import urlopen
    path = safe_path(path)
    def checksum(p):
        return git_blob_digest(p.read_bytes()) if kind == 'git' else file_digest(p)
    if path.exists():
        if not path.is_file() or path.stat().st_size > max_bytes or checksum(path) != expected:
            raise ValueError('Cached asset changed; preserve it and use a fresh owned runtime: ' + str(path))
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.' + uuid.uuid4().hex + '.partial')
    try:
        with urlopen(url, timeout=180) as response, temporary.open('xb') as stream:
            if not response.geturl().startswith('https://'):
                raise ValueError('Asset redirect must remain HTTPS')
            total = 0
            while True:
                data = response.read(1024 * 1024)
                if not data:
                    break
                total += len(data)
                if total > max_bytes:
                    raise ValueError('Asset exceeds bounded download size')
                stream.write(data)
        if checksum(temporary) != expected:
            raise ValueError('Pinned model/source asset checksum mismatch: ' + path.name)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)
    return path


def validate_alphabet(vocab):
    missing = sorted(set(VI_REQUIRED) - set(vocab.chars))
    if missing:
        raise RuntimeError('Recognizer cannot represent Vietnamese: ' + ''.join(missing))
    if vocab.decode(vocab.encode(VI_REQUIRED)) != VI_REQUIRED:
        raise RuntimeError('Vietnamese vocabulary roundtrip failed')
    return dict(required=len(set(VI_REQUIRED)), missing=[], roundtrip=True)


def public_record(record):
    # Raw lines/probabilities remain in technical checkpoints. This configurable
    # filter is not a calibrated accuracy guarantee or negative image evidence.
    lines = record['lines'] if record['status'] != 'error' else []
    text = '\n'.join(line['text'] for line in lines if line['text'].strip()
                     and line['confidence'] >= PUBLIC_MIN_CONFIDENCE
                     and not line.get('truncated', False) and line.get('public_eligible', True))
    video = record['video_id']
    return dict(video_id=video, frame_id=None, keyframe_n=record['keyframe_n'],
                image_path=f'data/{video.split("_V")[0]}/{video}/{Path(record["relative_path"]).name}',
                ocr_text=text, retrieval_text=text,
                status='error' if record['status'] == 'error' else 'ok' if text else 'empty')


def recognition_width(width, height):
    if min(width, height) < 1:
        raise ValueError('Empty detector crop')
    return max(32, math.ceil((32 * width / height) / 10) * 10)


def plan_line_segments(bgr):
    """Partition only at sufficiently wide background gaps; never cut by text guesses.

    Ranges are adjacent, nonoverlapping, and cover every original crop column.
    No arbitrary glyph cut, OCR-string overlap deduplication, or image synthesis.
    This geometric heuristic is not proof that a boundary is a word space.
    """
    import numpy as np
    value = np.asarray(bgr)
    if value.dtype != np.uint8 or value.ndim != 3 or value.shape[2] != 3:
        raise ValueError('Expected uint8 BGR crop')
    h, w = value.shape[:2]
    if recognition_width(w, h) <= OPTIONS['line_segment_trigger_width']:
        return dict(status='short', ranges=[[0, w]])
    # Otsu luminance threshold, no OpenCV/Paddle import in the Torch process.
    gray = np.rint(value @ np.array([.114, .587, .299])).astype(np.uint8)
    hist = np.bincount(gray.ravel(), minlength=256).astype(float)
    weight = np.cumsum(hist)
    moment = np.cumsum(hist * np.arange(256))
    denominator = weight * (weight[-1] - weight)
    score = np.zeros(256)
    np.divide((moment[-1] * weight - moment * weight[-1]) ** 2,
              denominator, out=score, where=denominator > 0)
    threshold = int(score.argmax())
    if not score.max():
        return dict(status='no_safe_gap', ranges=[], threshold=threshold)
    # The median border luminance estimates background polarity. Both light
    # ticker text on dark and dark print on light backgrounds are supported.
    border = np.concatenate([gray[0], gray[-1], gray[:, 0], gray[:, -1]])
    light_text = bool(np.median(border) <= threshold)
    ink = gray > threshold if light_text else gray <= threshold
    empty_columns = ~ink.any(axis=0)
    minimum_gap = max(2, math.ceil(h * OPTIONS['line_segment_min_gap_height_ratio']))
    edges = np.diff(np.r_[False, empty_columns, False].astype(np.int8))
    gaps = [(int(a), int(b)) for a, b in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1))
            if b - a >= minimum_gap and a > 0 and b < w]
    # Round down to a multiple of 10 so recognition_width's rounding cannot
    # push a planned segment beyond the configured normalized-width target.
    max_pixels = int((OPTIONS['line_segment_max_width'] // 10 * 10) * h // 32)
    ranges, start = [], 0
    while w - start > max_pixels:
        candidates = [(a + b) // 2 for a, b in gaps if start < (a + b) // 2 <= start + max_pixels]
        if not candidates or len(ranges) >= OPTIONS['line_segment_max_count'] - 1:
            return dict(status='no_safe_gap', ranges=[], threshold=threshold, light_text=light_text)
        end = max(candidates)
        ranges.append([start, end])
        start = end
    ranges.append([start, w])
    return dict(status='split', ranges=ranges, threshold=threshold, light_text=light_text,
                boundary_kind='background_gap_heuristic', overlap_px=0)


def pack_array(array):
    import base64
    import numpy as np
    value = np.asarray(array)
    if value.dtype != np.uint8 or value.ndim != 3 or value.shape[2] != 3 or value.nbytes > 48 * 1024 ** 2:
        raise ValueError('Expected bounded uint8 BGR image/crop')
    stream = io.BytesIO()
    np.save(stream, value, allow_pickle=False)
    return base64.b64encode(stream.getvalue()).decode('ascii')


def unpack_array(encoded_array):
    import base64
    import numpy as np
    if not isinstance(encoded_array, str) or len(encoded_array) > 68 * 1024 ** 2:
        raise ValueError('Detector image payload exceeds limit')
    value = np.load(io.BytesIO(base64.b64decode(encoded_array, validate=True)), allow_pickle=False)
    if value.dtype != np.uint8 or value.ndim != 3 or value.shape[2] != 3 or min(value.shape) < 1 or value.nbytes > 48 * 1024 ** 2:
        raise ValueError('Invalid detector image/crop')
    return value


def detector_worker():
    """Paddle-only child. Redirect ALL backend stdout to logs; JSON RPC on pipe."""
    from contextlib import redirect_stdout
    # Duplicate the protocol pipe first, then redirect FD1 as well as Python
    # stdout. Native Paddle/CUDA prints must not corrupt JSON replies.
    wire = os.fdopen(os.dup(sys.stdout.fileno()), 'w', encoding='utf-8', buffering=1)
    os.dup2(sys.stderr.fileno(), sys.stdout.fileno())
    with redirect_stdout(sys.stderr):
        import numpy as np
        import paddle
        from paddleocr import TextDetection
        from paddlex.inference.pipelines.components import CropByPolys, SortQuadBoxes
        gpu = require_gpu(paddle)
        versions = {name: importlib.metadata.version(name) for name in PADDLE_PINS}
        if versions != PADDLE_PINS:
            raise RuntimeError('Paddle child core version mismatch: ' + str(versions))
        root = safe_path(Path.home() / 'models-vietocr-pinned/detector')
        for name, (kind, expected) in DET_ASSETS.items():
            fetch_asset(DET_BASE + name, root / name, kind, expected, 100 * 1024 ** 2)
        detector = TextDetection(model_name=MODELS['detection'], model_dir=str(root), device='gpu:0',
            enable_hpi=False, use_tensorrt=False, limit_side_len=OPTIONS['detector_limit_side_len'],
            limit_type='max', thresh=.3, box_thresh=.6, unclip_ratio=1.5)
        actual = detector.paddlex_predictor
        if actual.model_name != MODELS['detection'] or actual.pp_option.device_type != 'gpu' or actual.pp_option.device_id != 0:
            raise RuntimeError('Detector model/device mismatch')
        crop, sort = CropByPolys(det_box_type='quad'), SortQuadBoxes()
        profile = dict(gpu, core_versions=versions,
                       model_files_sha256={name: file_digest(root / name) for name in DET_ASSETS})
        print(json.dumps(dict(status='ready', profile=profile)), file=wire, flush=True)
        for line in sys.stdin:
            if len(line) > 68 * 1024 ** 2:
                raise ValueError('Oversized detector request')
            request = json.loads(line)
            if request.get('action') != 'detect' or type(request.get('id')) is not int:
                raise ValueError('Invalid detector request')
            image = unpack_array(request['image'])
            results = list(detector.predict(input=image))
            if len(results) != 1 or 'error' in results[0]:
                raise RuntimeError('Unexpected detector response')
            # PaddleX SortQuadBoxes returns a Python list in this API. Normalize
            # either supported form once, preserving order for crops and JSON.
            polygons = np.asarray(sort(np.asarray(results[0]['dt_polys'])))
            crops = list(crop(image, polygons)) if len(polygons) else []
            if len(crops) != len(polygons):
                raise RuntimeError('Detector/crop alignment mismatch')
            print(json.dumps(dict(status='ok', id=request['id'], polygons=polygons.tolist(),
                                  crops=[pack_array(c) for c in crops])), file=wire, flush=True)


class DetectorProcess:
    """Persistent child: no GPU DLLs or Python package paths shared with Torch."""
    def __init__(self):
        import atexit
        import subprocess
        python = safe_path(os.environ['KIS_VI_PADDLE_PYTHON'])
        launcher = safe_path(os.environ['KIS_VI_LAUNCHER'])
        self.process = subprocess.Popen([str(python), '-I', '-S', str(launcher), 'worker',
            str(safe_path(__file__)), 'detector-worker'], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
            stderr=None, text=True, encoding='utf-8', bufsize=1)
        self.sequence = 0
        try:
            reply = self.read()
            if reply.get('status') != 'ready':
                raise RuntimeError('Detector child did not become ready')
            self.profile = reply['profile']
        except BaseException:
            self.close()
            raise
        atexit.register(self.close)

    def read(self):
        line = self.process.stdout.readline(68 * 1024 ** 2 + 1)
        if not line or len(line) > 68 * 1024 ** 2:
            raise RuntimeError('Paddle child failed or sent invalid response; inspect its stderr above')
        return json.loads(line)

    def detect(self, image):
        self.sequence += 1
        self.process.stdin.write(json.dumps(dict(action='detect', id=self.sequence, image=pack_array(image))) + '\n')
        self.process.stdin.flush()
        reply = self.read()
        if reply.get('status') != 'ok' or reply.get('id') != self.sequence:
            raise RuntimeError('Detector response identity/status mismatch')
        polygons, encoded_crops = reply['polygons'], reply['crops']
        if not isinstance(polygons, list) or not isinstance(encoded_crops, list) or len(polygons) != len(encoded_crops):
            raise RuntimeError('Detector crop identity/count mismatch')
        return polygons, [unpack_array(c) for c in encoded_crops]

    def close(self):
        import subprocess
        process = self.process
        if process.stdin and not process.stdin.closed:
            try:
                process.stdin.close()
            except OSError:
                pass
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()  # Only this adapter's child, never another runtime.
            process.wait(timeout=15)
        if process.stdout:
            process.stdout.close()


class Adapter:
    def __init__(self):
        import numpy as np
        versions = {name: importlib.metadata.version(name) for name in PINS}
        if versions != PINS:
            raise RuntimeError(f'Core version mismatch: {versions}; expected {PINS}')
        import torch
        import yaml
        from PIL import Image
        if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
            raise RuntimeError('VietOCR requires the same visible GPU0, no CPU fallback')
        smoke = torch.tensor([[2.]], device='cuda:0')
        if float((smoke @ smoke).item()) != 4.:
            raise RuntimeError('Torch CUDA smoke failed')
        torch.set_num_threads(2)
        root = safe_path(Path.home() / 'models-vietocr-pinned')
        root.mkdir(parents=True, exist_ok=True)
        source = root / 'source'
        for name, expected in VI_BLOBS.items():
            fetch_asset(f'https://raw.githubusercontent.com/pbcquoc/vietocr/{VI_REVISION}/{name}',
                        source / name, 'git', expected, 1024 * 1024)
        weights = fetch_asset(WEIGHTS_URL, root / 'vgg_transformer.pth', 'sha256', WEIGHTS_SHA256, 200 * 1024 ** 2)
        detector = root / 'detector'
        for name, (kind, expected) in DET_ASSETS.items():
            fetch_asset(DET_BASE + name, detector / name, kind, expected, 100 * 1024 ** 2)
        sys.path.insert(0, str(source))
        from vietocr.model.transformerocr import VietOCR
        from vietocr.model.vocab import Vocab
        config = yaml.safe_load((source / 'config/base.yml').read_text(encoding='utf-8'))
        config.update(yaml.safe_load((source / 'config/vgg-transformer.yml').read_text(encoding='utf-8')))
        config['cnn']['pretrained'] = False  # NO unrelated ImageNet download.
        if config['backbone'] != 'vgg19_bn' or config['seq_modeling'] != 'transformer':
            raise RuntimeError('Unexpected VietOCR architecture')
        self.vocab = Vocab(config['vocab'])
        alphabet = validate_alphabet(self.vocab)
        model = VietOCR(len(self.vocab), config['backbone'], config['cnn'], config['transformer'], 'transformer')
        # Pinned tensor-only state, strict keys/shapes/vocabulary. NEVER unsafe
        # weights_only=False fallback for an incompatible legacy pickle.
        state = torch.load(weights, map_location='cpu', weights_only=True)
        model.load_state_dict(state, strict=True)
        if model.transformer.fc.out_features != len(self.vocab):
            raise RuntimeError('Checkpoint/vocabulary head mismatch')
        del state
        self.model = model.to('cuda:0').eval()
        self.detector = DetectorProcess()
        self.config = config
        self.engine = dict(device='gpu:0', gpu_name=torch.cuda.get_device_name(0), models=MODELS,
            core_versions=versions, alphabet=alphabet, detector_process=self.detector.profile,
            runtime_isolation='separate venvs/processes; incompatible NVIDIA dependencies never coinstalled',
            source_revision=VI_REVISION, weights_source=WEIGHTS_URL,
            model_files_sha256={'recognition.pth': file_digest(weights),
                **{'detector/' + n: file_digest(detector / n) for n in DET_ASSETS}},
            source_files_sha256={name: file_digest(source / name) for name in VI_BLOBS},
            recognition_config=config,
            confidence='single crop: mean character softmax; segmented line: minimum segment mean; both uncalibrated',
            warnings=['Alphabet support and CUDA smoke do not establish OCR accuracy.',
                      'VietOCR can hallucinate on blurred/background crops. Inspect preview before bulk.',
                      'Weights mirrored by doanhm; origin/model compatibility checked by pinned digest and strict state load.'])
        print(json.dumps(self.engine, ensure_ascii=False, indent=2), flush=True)

    def recognize_crop(self, bgr):
        return self.recognize_crops([bgr])[0]

    def recognize_crops(self, crops):
        """Real tensor microbatches, exact resized-width buckets, original order.

        No extra width padding or mixed precision: preserve v3 input pixels.
        Width buckets and final tails may contain fewer than32 real crops.
        """
        import numpy as np
        import torch
        from PIL import Image
        batch_size = OPTIONS['recognition_batch_size']
        if type(batch_size) is not int or not 1 <= batch_size <= 32:
            raise ValueError('Recognition batch must be an integer from1 to32')
        buckets, shapes = {}, []
        for i, crop in enumerate(crops):
            value = np.asarray(crop)
            if value.dtype != np.uint8 or value.ndim != 3 or value.shape[2] != 3:
                raise ValueError('Expected uint8 BGR recognition crop')
            h, w = value.shape[:2]
            desired = recognition_width(w, h)
            width = min(OPTIONS['recognition_max_width'], desired)
            shapes.append((w, h, width, desired))
            buckets.setdefault(width, []).append(i)
        results = [None] * len(crops)
        if not hasattr(self, 'batch_stats'):
            self.batch_stats = dict(forward_batches=0, crops=0, batch_size_histogram={})
        for width, indices in buckets.items():
            for start in range(0, len(indices), batch_size):
                chosen = indices[start:start + batch_size]
                tensors = []
                for index in chosen:
                    image = Image.fromarray(crops[index][:, :, ::-1]).resize((width, 32), Image.Resampling.LANCZOS)
                    tensors.append(torch.from_numpy(np.asarray(image).transpose(2, 0, 1).copy()).float().div(255))
                try:
                    tensor = torch.stack(tensors).to('cuda:0')
                    decoded = self.decode_batch(tensor)
                except torch.cuda.OutOfMemoryError as exc:
                    raise RuntimeError('VietOCR batch32 CUDA OOM; run stopped, checkpoints preserved. '
                                       'No silent smaller-batch/CPU fallback; inspect GPU memory before retry.') from exc
                histogram = self.batch_stats['batch_size_histogram']
                histogram[str(len(chosen))] = histogram.get(str(len(chosen)), 0) + 1
                self.batch_stats['forward_batches'] += 1
                self.batch_stats['crops'] += len(chosen)
                for index, row in zip(chosen, decoded):
                    w, h, width, desired = shapes[index]
                    results[index] = dict(row, width_capped=desired > width, crop_size=[w, h],
                        recognizer_size=[width, 32], probability_kind='uncalibrated_mean_character_softmax',
                        completeness='not_verified', inference_batch_size=len(chosen),
                        inference_batch_width=width, inference_batch_policy=OPTIONS['recognition_batch_policy'])
        return results

    def decode_batch(self, tensor):
        import torch
        count = tensor.shape[0]
        ids = [[self.vocab.go] for _ in range(count)]
        probabilities = [[] for _ in range(count)]
        finished = [False] * count
        active = list(range(count))
        with torch.inference_mode():
            memory = self.model.transformer.forward_encoder(self.model.cnn(tensor))
            for _ in range(OPTIONS['recognition_max_tokens']):
                tokens = torch.tensor([ids[i] for i in active], dtype=torch.long, device='cuda:0').transpose(0, 1)
                logits, _ = self.model.transformer.forward_decoder(tokens, memory)
                distribution = torch.softmax(logits[:, -1, :], dim=-1)
                if distribution.shape[0] != len(active):
                    raise RuntimeError('VietOCR decoder batch alignment mismatch')
                if not bool(torch.isfinite(distribution).all().item()):
                    raise RuntimeError('Non-finite recognizer output')
                scores, choices = distribution.max(dim=-1)
                # One batched host transfer per vector, not a .item() sync for
                # every crop. Finished rows are removed from memory and tokens.
                keep, remaining = [], []
                for column, (identity, token, score) in enumerate(zip(active, choices.tolist(), scores.tolist())):
                    ids[identity].append(token)
                    if token == self.vocab.eos:
                        finished[identity] = True
                    else:
                        if token < 4:
                            raise RuntimeError('Unexpected non-text token; no guessed substitution')
                        probabilities[identity].append(score)
                        keep.append(column)
                        remaining.append(identity)
                if not remaining:
                    break
                if len(keep) != len(active):
                    memory = memory[:, keep, :]
                active = remaining
        return [dict(text=self.vocab.decode(ids[i]),
                     confidence=sum(probabilities[i]) / len(probabilities[i]) if probabilities[i] else 0.,
                     truncated=not finished[i], decoder_eos=finished[i], decoded_tokens=len(probabilities[i]))
                for i in range(count)]

    def recognize_line(self, bgr):
        return self.recognize_lines([bgr])[0]

    def recognize_lines(self, crops):
        jobs, descriptors = [], []
        for bgr in crops:
            plan = plan_line_segments(bgr)
            full_index = len(jobs)
            jobs.append(bgr)
            segment_indices = []
            if plan['status'] == 'split':
                for start, end in plan['ranges']:
                    segment_indices.append((len(jobs), start, end))
                    jobs.append(bgr[:, start:end].copy())
            descriptors.append((bgr, plan, full_index, segment_indices))
        raw = self.recognize_crops(jobs)
        if len(raw) != len(jobs):
            raise RuntimeError('Recognition job count mismatch')
        results = []
        for bgr, plan, full_index, indices in descriptors:
            segments = [dict(raw[index], x_range_crop=[start, end],
                             crop_bgr_sha256=digest(jobs[index].tobytes())) for index, start, end in indices]
            results.append(self.assemble_line(bgr, raw[full_index], plan, segments))
        return results

    def assemble_line(self, bgr, full, plan, segments):
        common = dict(segmentation=plan, completeness='not_verified')
        if plan['status'] == 'short':
            return dict(full, **common, recognition_strategy='full_line', public_eligible=True)
        if plan['status'] != 'split':
            # Keep raw baseline for inspection, but do not silently publish a
            # known-risk long prefix when no safe geometric partition exists.
            return dict(full, **common, full_line=full, recognition_strategy='full_line_unsegmented',
                        public_eligible=False, review_reasons=['long_line_no_safe_gap'])
        reasons = []
        if any(not s['text'].strip() for s in segments):
            reasons.append('segment_empty')
        if any(s['confidence'] < PUBLIC_MIN_CONFIDENCE for s in segments):
            reasons.append('segment_low_confidence')
        if any(s['truncated'] for s in segments):
            reasons.append('segment_token_cap')
        if any(s['width_capped'] for s in segments):
            reasons.append('segment_width_cap')
        # No language correction or fuzzy overlap merge: real repeated words
        # survive. Choice is geometric, never based on greater confidence.
        return dict(text=' '.join(s['text'].strip() for s in segments),
                    confidence=min(s['confidence'] for s in segments),
                    truncated=any(s['truncated'] for s in segments),
                    width_capped=any(s['width_capped'] for s in segments),
                    crop_size=list(reversed(bgr.shape[:2])), recognizer_size=full['recognizer_size'],
                    probability_kind='minimum_of_uncalibrated_segment_means',
                    decoder_eos=all(not s['truncated'] for s in segments),
                    recognition_strategy='word_gap_segments', public_eligible=not reasons,
                    review_reasons=reasons, full_line=full, segments=segments, **common)

    def recognize(self, padded):
        return self.recognize_many([padded])[0]

    def recognize_many(self, padded_images):
        import numpy as np
        jobs, frames = [], []
        for padded in padded_images:
            bgr = np.asarray(padded)[:, :, ::-1].copy()
            polygons, crops = self.detector.detect(bgr)
            if len(crops) != len(polygons):
                raise RuntimeError('Detector/crop alignment mismatch')
            frames.append((len(jobs), polygons))
            jobs.extend(crops)
        recognized = self.recognize_lines(jobs)
        return [[dict(recognized[start + i], polygon_padded=polygon) for i, polygon in enumerate(polygons)]
                for start, polygons in frames]


_legacy_record_for = record_for
def record_for(item, sha, profile_hash, decoded, error, adapter):
    # Reuse geometry/identity validation, preserve extra per-line diagnostics.
    if error:
        return _legacy_record_for(item, sha, profile_hash, decoded, error, adapter)
    captured = []
    class Capture:
        def recognize(self, image):
            captured.extend(adapter.recognize(image))
            return captured
    record = _legacy_record_for(item, sha, profile_hash, decoded, None, Capture())
    for target, raw in zip(record['lines'], captured):
        for name in ('truncated', 'width_capped', 'crop_size', 'recognizer_size', 'probability_kind',
                     'decoder_eos', 'decoded_tokens', 'completeness', 'recognition_strategy',
                     'inference_batch_size', 'inference_batch_width', 'inference_batch_policy',
                     'public_eligible', 'review_reasons', 'segmentation', 'segments', 'full_line'):
            if name in raw:
                target[name] = raw[name]
        target['public_included'] = (bool(target['text'].strip()) and target['confidence'] >= PUBLIC_MIN_CONFIDENCE
                                     and not target.get('truncated', False) and target.get('public_eligible', True))
    return record


def records_for_batch(rows, profile_hash, adapter):
    good = [i for i, row in enumerate(rows) if row[3] is None]
    padded = [rows[i][2][0] for i in good]
    if hasattr(adapter, 'recognize_many'):
        values = adapter.recognize_many(padded) if padded else []
    else:
        values = [adapter.recognize(image) for image in padded]
    if len(values) != len(good):
        raise RuntimeError('Frame batch reply count mismatch')
    by_index = dict(zip(good, values))
    class PreparedFrame:
        def __init__(self, lines):
            self.lines = lines
        def recognize(self, image):
            return self.lines
    return [record_for(item, sha, profile_hash, decoded, error,
                       PreparedFrame(by_index.get(i, [])))
            for i, (item, sha, decoded, error) in enumerate(rows)]


_legacy_run = run
def run(root, output, env_root, run_id, adapter=None, progress=None):
    """Buffer up to32 uncached frames to fill width-bucketed crop batches.

    Detector still runs one image/request. Checkpoints retain individual image
    hashes and order; no previous profile is reused when batching changes.
    """
    # Preserve the lightweight test/custom adapter interface. Production Adapter
    # always uses recognize_many and never silently switches to scalar inference.
    if adapter is not None and not hasattr(adapter, 'recognize_many'):
        return _legacy_run(root, output, env_root, run_id, adapter, progress)
    root, output, env_root = validate_paths(root, output, env_root)
    claim_output(root, output)
    state = dict(run_id=run_id, phase='running', complete=False)
    invalidate_ready(output)
    atomic_json(output / 'run.json', state)
    try:
        items = inventory(root)
        adapter = adapter if adapter is not None else Adapter()
        profile, profile_hash = make_profile(adapter)
        smoke = None
        for item in items:
            raw = safe_path(root / item['relative_path']).read_bytes()
            decoded, error = decode_or_error(raw)
            if not error:
                smoke = record_for(item, digest(raw), profile_hash, decoded, None, adapter)
                break
        if smoke is None:
            raise RuntimeError('No decodable real image for mandatory GPU OCR smoke')
        print(f'Real image OCR smoke OK: {smoke["relative_path"]}; profile={profile_hash}', flush=True)
        atomic_json(output / 'profile.json', profile)
        if progress is None:
            from tqdm import tqdm
            progress = lambda rows: tqdm(rows, total=len(rows), unit='image',
                desc='OCR batch32 (including resume checks)', file=sys.stdout, mininterval=1, smoothing=.1)
        window = OPTIONS['recognition_frame_window']
        if type(window) is not int or not 1 <= window <= 32:
            raise ValueError('Recognition frame window must be from1 to32')
        pending = []
        def flush():
            if not pending:
                return
            records = records_for_batch([(item, sha, decoded, None) for item, sha, decoded in pending],
                                        profile_hash, adapter)
            for record in records:
                save_checkpoint(output, record)
            pending.clear()
        errors, skipped = 0, 0
        for item in progress(items):
            raw = safe_path(root / item['relative_path']).read_bytes()
            sha = digest(raw)
            old = cached(output, item, sha, profile_hash)
            if old and old['status'] in {'ok', 'empty'}:
                skipped += 1
                continue
            if item['id'] == smoke['id'] and sha == smoke['source_sha256']:
                save_checkpoint(output, smoke)
                continue
            decoded, error = decode_or_error(raw)
            if error:
                record = record_for(item, sha, profile_hash, None, error, adapter)
                save_checkpoint(output, record)
                errors += 1
                print(f'Image decode error: {item["relative_path"]}: {record["error"]}', flush=True)
            else:
                pending.append((item, sha, decoded))
                if len(pending) >= window:
                    flush()
        flush()
        state.update(phase='finished', complete=errors == 0, errors=errors, skipped=skipped,
                     total=len(items), profile_sha256=profile_hash, inventory=items,
                     recognition_batches=getattr(adapter, 'batch_stats', None))
        atomic_json(output / 'run.json', state)
        print(f'Finished: {len(items)} images; resumed {skipped}; decode errors {errors}', flush=True)
        print('Actual recognition batches: ' + json.dumps(state['recognition_batches']), flush=True)
    except BaseException as exc:
        state.update(phase='fatal', complete=False, error=f'{type(exc).__name__}: {exc}')
        atomic_json(output / 'run.json', state)
        traceback.print_exc()
        raise


def select_preview_items(items, count, identities=None):
    if type(count) is not int or not 1 <= count <= 50:
        raise ValueError('Preview count must be in [1,50]')
    if identities is None:
        identities = []
    if (not isinstance(identities, list) or len(identities) > 50
            or any(not isinstance(v, str) or not re.fullmatch(r'L\d{2}_V\d{3}:[1-9]\d*', v) for v in identities)
            or len(set(identities)) != len(identities)):
        raise ValueError('Preview identities must be a unique list of at most50 canonical VIDEO:n values')
    if identities:
        lookup = {item['id']: item for item in items}
        if any(v not in lookup for v in identities):
            raise ValueError('Requested preview identity absent from INPUT_ROOT; no fallback')
        return [lookup[v] for v in identities]
    count = min(count, len(items))
    return [items[(i * (len(items) - 1)) // max(1, count - 1)] for i in range(count)]


def preview(root, output, env_root, run_id, count, identities=None):
    from PIL import Image, ImageDraw
    root, output, env_root = validate_paths(root, output, env_root)
    claim_output(root, output)
    invalidate_ready(output)
    items = inventory(root)
    selected = select_preview_items(items, count, identities)
    adapter = Adapter()
    profile, profile_hash = make_profile(adapter)
    destination = output / 'previews' / run_id
    destination.mkdir(parents=True, exist_ok=False)
    records = []
    window = OPTIONS['recognition_frame_window']
    for start in range(0, len(selected), window):
        rows = []
        for item in selected[start:start + window]:
            raw = safe_path(root / item['relative_path']).read_bytes()
            decoded, error = decode_or_error(raw)
            rows.append((item, digest(raw), decoded, error))
        batch = records_for_batch(rows, profile_hash, adapter)
        for offset, (record, (_, _, decoded, error)) in enumerate(zip(batch, rows)):
            record['public_projection'] = public_record(record)
            if not error:
                image = decoded[0].copy()
                draw = ImageDraw.Draw(image)
                for i, line in enumerate(record['lines']):
                    points = [tuple(p) for p in line['polygon_padded']]
                    draw.line(points + points[:1], fill='lime' if line['public_included'] else 'red', width=2)
                    draw.text(points[0], str(i + 1), fill='yellow')
                image.save(destination / f'{start + offset:02d}.png')
            records.append(record)
    atomic_json(destination / 'preview.json', dict(schema=SCHEMA, profile=profile, records=records,
        recognition_batches=getattr(adapter, 'batch_stats', None),
        note='GPU/alphabet smoke is not an accuracy test. Review raw lines and boxes; no preview checkpoint is reused by bulk.'))
    atomic_json(output / 'latest-preview.json', dict(run_id=run_id, directory=str(destination)))
    print('Preview ready: ' + str(destination), flush=True)


_legacy_export = export_current
def export_current(root, output, env_root, run_id, archive='/kaggle/working/ocr_vietnamese_v3_batch32.zip'):
    root, output, env_root = validate_paths(root, output, env_root)
    archive = safe_path(archive)
    technical = archive.with_name(archive.stem + '_diagnostics.zip')
    if archive == technical:
        raise ValueError('Public and technical archives must be distinct')
    receipt = output / 'technical-publication.json'
    if technical.exists() and (not receipt.exists() or read_json(receipt).get('sha256') != file_digest(technical)):
        raise PublicationCollision('Unowned/changed technical ZIP; refusing overwrite')
    destination = _legacy_export(root, output, env_root, run_id, archive)
    temporary = technical.with_name('.ocr-vi-diagnostics-' + uuid.uuid4().hex + '.tmp')
    try:
        state = read_json(output / 'run.json')
        with zipfile.ZipFile(temporary, 'x', zipfile.ZIP_DEFLATED) as z:
            z.write(output / 'profile.json', 'profile.json')
            z.write(output / 'run.json', 'run.json')
            for item in state['inventory']:
                sha = file_digest(safe_path(root / item['relative_path']))
                record = cached(output, item, sha, state['profile_sha256'])
                if record is None:
                    raise RuntimeError('Technical checkpoint changed during export')
                name = f'checkpoints/{item["video_id"].split("_V")[0]}/{item["video_id"]}/{Path(item["relative_path"]).stem}.json'
                z.writestr(name, encoded(record))
        with zipfile.ZipFile(temporary) as z:
            if z.testzip() is not None:
                raise RuntimeError('Technical ZIP CRC failure')
        os.replace(temporary, technical)
        atomic_json(receipt, dict(run_id=run_id, archive=str(technical), sha256=file_digest(technical)))
    except BaseException:
        invalidate_current_ready(root, output, env_root, run_id)
        raise
    finally:
        temporary.unlink(missing_ok=True)
    return destination


def main():
    if sys.argv[1:] == ['detector-worker']:
        detector_worker()
        return
    parser = argparse.ArgumentParser()
    parser.add_argument('action', choices=['preview', 'run', 'export'])
    for name in ('input_root', 'output', 'env_root', 'run_id'):
        parser.add_argument(name)
    parser.add_argument('--archive', default='/kaggle/working/ocr_vietnamese_v3_batch32.zip')
    parser.add_argument('--preview-count', type=int, default=8)
    parser.add_argument('--preview-identities', default='[]')
    args = parser.parse_args()
    if not re.fullmatch(r'[a-f0-9]{32}', args.run_id):
        raise ValueError('Invalid invocation ID')
    validate_publish(args.input_root, args.output, args.env_root, args.archive)
    if args.action == 'preview':
        preview(args.input_root, args.output, args.env_root, args.run_id, args.preview_count,
                json.loads(args.preview_identities))
    elif args.action == 'run':
        output = safe_path(args.output)
        latest = read_json(output / 'latest-preview.json')
        preview_path = safe_path(latest['directory'])
        if preview_path.parent != output / 'previews':
            raise ValueError('Preview path outside owned output')
        expected = read_json(preview_path / 'preview.json')['profile']
        adapter = Adapter()
        if make_profile(adapter)[1] != digest(encoded(expected)):
            raise RuntimeError('Model/settings changed since preview; preview again before bulk')
        run(args.input_root, args.output, args.env_root, args.run_id, adapter=adapter)
    else:
        export_current(args.input_root, args.output, args.env_root, args.run_id, args.archive)


if __name__ == '__main__':
    main()
'''
LAUNCHER_SOURCE = '"""Run an owned pip-less venv without Kaggle\'s global sitecustomize hook.\n\nInvoked as: <venv-python> -I -S launcher.py {pip <host-pip-source>|worker <script>} ...\nOnly that venv\'s site-packages are enabled. pip source is imported as ONE package,\nnot by adding the host site-packages directory. No dependencies are suppressed.\n"""\n\nimport importlib.util\nimport os\nfrom pathlib import Path\nimport runpy\nimport site\nimport sys\n\n\ndef main():\n    if not sys.flags.no_site or not sys.flags.isolated:\n        raise RuntimeError(\'Launcher requires both -I and -S\')\n    prefix = Path(os.path.abspath(sys.executable)).parent.parent\n    if not (prefix / \'pyvenv.cfg\').is_file():\n        raise RuntimeError(\'Launcher must use an explicit owned venv interpreter\')\n    # Python <=3.13 sets venv prefix in site.main(); -S skips it. Restore ONLY\n    # the explicit venv prefix, then add its packages without calling site.main.\n    sys.prefix = sys.exec_prefix = str(prefix)\n    packages = (prefix / \'Lib/site-packages\' if os.name == \'nt\' else\n                prefix / f\'lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages\')\n    if not packages.is_dir():\n        raise RuntimeError(\'Target venv site-packages missing\')\n    sys.path[:] = [p for p in sys.path if \'site-packages\' not in p and \'dist-packages\' not in p]\n    site.addsitedir(str(packages))  # Local .pth files only; no global sitecustomize.\n    if len(sys.argv) < 3 or sys.argv[1] not in {\'pip\', \'worker\'}:\n        raise ValueError(\'Expected pip source directory or worker script\')\n    mode, source, arguments = sys.argv[1], Path(sys.argv[2]), sys.argv[3:]\n    if mode == \'pip\':\n        spec = importlib.util.spec_from_file_location(\'pip\', source / \'__init__.py\',\n                                                     submodule_search_locations=[str(source)])\n        if spec is None or spec.loader is None:\n            raise RuntimeError(\'Host pip package source unavailable\')\n        module = importlib.util.module_from_spec(spec)\n        sys.modules[\'pip\'] = module\n        spec.loader.exec_module(module)\n        sys.argv = [\'pip\', *arguments]\n        runpy.run_module(\'pip\', run_name=\'__main__\')\n    else:\n        sys.argv = [str(source), *arguments]\n        runpy.run_path(str(source), run_name=\'__main__\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
"""Notebook setup fragment; appended after WORKER_SOURCE/LAUNCHER_SOURCE literals."""

# Load stdlib helpers only. Models run in separate children, never in the kernel.
helpers = {'__name__': 'notebook_helpers'}
exec(compile(WORKER_SOURCE, '<visible-worker>', 'exec'), helpers)
root = helpers['safe_path'](INPUT_ROOT)
ARCHIVE = Path('/kaggle/working/ocr_vietnamese_v3_batch32.zip')
helpers['inventory'](root)
if sys.version_info[:2] != (3, 13) or sys.platform != 'linux':
    raise RuntimeError('This notebook requires Kaggle Linux Python 3.13')
SCRATCH, OUTPUT, ENV_ROOT = helpers['select_scratch'](root, ARCHIVE)
helpers['invalidate_ready'](OUTPUT)
child_env = helpers['isolated_environment'](SCRATCH, os.environ)
child_env.update(FLAGS_allocator_strategy='auto_growth', KIS_VI_OCR_MIN_CONFIDENCE=str(PUBLIC_MIN_CONFIDENCE))
BOOTSTRAP = SCRATCH / 'isolated-launcher.py'
WORKER = OUTPUT / 'worker.py'
helpers['atomic_bytes'](BOOTSTRAP, LAUNCHER_SOURCE.encode('utf-8'))
helpers['atomic_bytes'](WORKER, WORKER_SOURCE.encode('utf-8'))
import pip
PIP_SOURCE = Path(pip.__file__).resolve().parent  # Existing host pip, not its other packages.


def checked_step(command, label):
    print('\n' + label, flush=True)
    # Preserve installation progress in Save Version logs; dependency check
    # diagnostics must not be replaced by an opaque CalledProcessError alone.
    result = subprocess.run(command, env=child_env, cwd=SCRATCH, check=False)
    if result.returncode:
        raise RuntimeError(f'{label} failed (exit {result.returncode}); see complete pip output above. No dependency gate bypassed.')


def prepare_environment(path, pins, role):
    path = helpers['safe_path'](path)
    helpers['safe_runtime_tree'](path)
    if role == 'torch':
        marker, expected = helpers['runtime_identity'](path)
    else:
        marker = path / 'split-runtime.json'
        expected = dict(schema='vietnamese-paddle-runtime-v2', pins=pins)
        if path.exists() and any(path.iterdir()):
            if not marker.is_file() or helpers['read_json'](marker) != expected:
                raise RuntimeError('Unowned/incompatible Paddle venv; refusing modification')
    if not marker.exists():
        helpers['atomic_json'](marker, expected)
    python = path / 'bin/python'
    if not python.exists():
        checked_step([sys.executable, '-I', '-S', '-m', 'venv', '--without-pip', '--copies', str(path)],
                     f'Create {role} runtime (no ensurepip or global sitecustomize)')
    constraints = path / 'core-constraints.txt'
    helpers['atomic_bytes'](constraints, ('\n'.join(f'{k}=={v}' for k, v in pins.items()) + '\n').encode())
    prefix = [str(python), '-I', '-S', str(BOOTSTRAP), 'pip', str(PIP_SOURCE), '--isolated', '--disable-pip-version-check']
    if role == 'paddle':
        packages = [f'{k}=={v}' for k, v in pins.items() if k != 'paddlepaddle-gpu']
        packages += ['https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/paddlepaddle_gpu-3.2.2-cp313-cp313-linux_x86_64.whl']
        indexes = ['--index-url', 'https://pypi.org/simple']
    else:
        packages = [f'{k}=={v}' for k, v in pins.items()]
        indexes = ['--index-url', 'https://pypi.org/simple', '--extra-index-url', 'https://download.pytorch.org/whl/cu126']
    checked_step(prefix + ['install', '--only-binary=:all:', '--no-cache-dir', '--constraint', str(constraints),
                          *indexes, *packages], f'Install {role} dependencies ONLY into {path}')
    checked_step(prefix + ['check'], f'Validate {role} dependencies with pip check')
    return python


PADDLE_ENV_ROOT = SCRATCH / 'runtime-paddle'
PADDLE_PYTHON = prepare_environment(PADDLE_ENV_ROOT, helpers['PADDLE_PINS'], 'paddle')
PYTHON = prepare_environment(ENV_ROOT, helpers['PINS'], 'torch')
child_env.update(KIS_VI_PADDLE_PYTHON=str(PADDLE_PYTHON), KIS_VI_LAUNCHER=str(BOOTSTRAP))
WORKER_COMMAND = [str(PYTHON), '-I', '-S', str(BOOTSTRAP), 'worker', str(WORKER)]
print('Two separate runtimes validated. NVIDIA dependencies are NOT shared. GPU/model verification follows in preview.', flush=True)


## 3. Preview before bulk
        Samples are spread across the selected subtree unless PREVIEW_IDENTITIES explicitly selects exact VIDEO:n values. To retest diagnosed L01 cases use ['L01_V005:69', 'L01_V013:170', 'L01_V015:196', 'L01_V020:29', 'L01_V030:67']; use an additional unseen sample before bulk. Absent/duplicate requested identities fail, never silently substitute another image.

        Green boxes pass public filters, not accuracy verification. Red boxes are excluded and remain in diagnostics. Long crops whose normalized width exceeds512 are partitioned only at wide background gaps, targeting at most480px per segment. No overlap, fuzzy text correction, or confidence-based winner selection. Full-line baseline AND each segment remain in diagnostics. No safe gap or any blank/low-confidence/truncated segment excludes the whole long line rather than publishing a partial prefix. This may reduce coverage; inspect it.

        Detector settings/models/weights and v3 segmentation rules are unchanged. Crop jobs (full-line baselines and segments) are grouped by their exact normalized width and decoded in tensor batches up to32. Each crop has independent EOS/probabilities and results are restored to original frame/line order. EOS/truncated=false does NOT establish complete text; completeness always remains not_verified. Recheck text against the scalar v3 preview; batching can cause numerical differences on actual GPU.

In [ ]:
if helpers['safe_path'](INPUT_ROOT) != root or str(float(PUBLIC_MIN_CONFIDENCE)) != child_env['KIS_VI_OCR_MIN_CONFIDENCE']:
    raise RuntimeError('Input/confidence changed: rerun setup and preview first')
RUN_OK = False
EXPORT_OK = False
PREVIEW_ID = uuid.uuid4().hex
subprocess.run([*WORKER_COMMAND, 'preview', str(root), str(OUTPUT),
                str(ENV_ROOT), PREVIEW_ID, '--archive', str(ARCHIVE), '--preview-count', str(PREVIEW_COUNT),
                '--preview-identities', json.dumps(PREVIEW_IDENTITIES)],
               env=child_env, cwd=SCRATCH, check=True)
preview_root = Path(helpers['read_json'](OUTPUT / 'latest-preview.json')['directory'])
preview = helpers['read_json'](preview_root / 'preview.json')
print('Actual recognition batches:', preview.get('recognition_batches'))
from IPython.display import Image as DisplayImage, display
for i, record in enumerate(preview['records']):
    print(record['video_id'], record['keyframe_n'], 'status=', record['status'])
    image = preview_root / f'{i:02d}.png'
    if image.is_file():
        display(DisplayImage(filename=str(image), width=1000))
    for j, line in enumerate(record['lines']):
        print(j + 1, repr(line['text']), 'confidence=', round(line['confidence'], 4),
              'included=', line['public_included'], 'truncated=', line.get('truncated', False),
              'width_capped=', line.get('width_capped', False),
              'strategy=', line.get('recognition_strategy'), 'review=', line.get('review_reasons', []))
        if 'inference_batch_size' in line:
            print('  Actual tensor batch:', line['inference_batch_size'], 'width=', line['inference_batch_width'])
        if 'full_line' in line:
            print('  Full-line baseline:', repr(line['full_line']['text']))
        for segment in line.get('segments', []):
            print('  Segment', segment['x_range_crop'], repr(segment['text']),
                  'confidence=', round(segment['confidence'], 4), 'size=', segment['recognizer_size'],
                  'batch=', segment.get('inference_batch_size'))
print('Preview is NOT an accuracy benchmark. Inspect original words; do not correct source text by guessing.')


## 4. Full OCR / same-session resume
        Set `RUN_FULL=True` in cell1 after checking preview, then run this cell. Successful checkpoints are reused only with identical original-byte SHA256 and full model/config/source profile. No source image rename/overwrite, no `n-1` frame inference. Corrupt images have explicit error; model failures stop. Technical record keeps all raw line text, probabilities, polygons and per-image SHA; public text applies only the explicit quality filter.

In [ ]:
RUN_OK = False
EXPORT_OK = False
if not RUN_FULL:
    print('Bulk OCR skipped. Inspect preview, set RUN_FULL=True, then rerun this cell. No ZIP published.')
else:
    if helpers['safe_path'](INPUT_ROOT) != root or str(float(PUBLIC_MIN_CONFIDENCE)) != child_env['KIS_VI_OCR_MIN_CONFIDENCE']:
        raise RuntimeError('Input/confidence changed: rerun setup and preview first')
    RUN_OK = False
    RUN_ID = uuid.uuid4().hex
    EXPORT_OK = False
    try:
        helpers['validate_publish'](root, OUTPUT, ENV_ROOT, ARCHIVE)
        helpers['invalidate_ready'](OUTPUT)
        subprocess.run([*WORKER_COMMAND, 'run', str(root), str(OUTPUT),
                        str(ENV_ROOT), RUN_ID, '--archive', str(ARCHIVE)],
                       env=child_env, cwd=SCRATCH, check=True)
        RUN_OK = True
    except BaseException:
        helpers['invalidate_current_ready'](root, OUTPUT, ENV_ROOT, RUN_ID)
        RUN_OK = False
        print('WARNING: run failed; an older ocr.zip may remain, not a current result.', flush=True)
        raise


## 5. Download two separate archives
        `ocr_vietnamese_v3_batch32.zip`: canonical `Lxx/Lxx_Vxxx/NNN.json`, each a one-record array with exactly the same seven public fields as before. `frame_id=null`, `retrieval_text==ocr_text`. Missing/filtered/empty is unknown, not negative evidence.

        `ocr_vietnamese_v3_batch32_diagnostics.zip`: profile, pinned source/model hashes, geometry, raw full-line and segment results/ranges, actual tensor batch sizes, confidences, EOS/token-cap/width flags, review reasons and original image SHA. run.json includes the actual recognition batch histogram. Keep this for quality evaluation/provenance. Do not merge it into captions or blindly replace production OCR. Scalar v3 and batch16 ZIPs/checkpoints are preserved; batch32 uses a new scratch root and new archive names. An existing unrelated archive is refused.

        Resume works in the same Kaggle session; transient scratch disappears on session reset. Snapshot/export logs are not proof of OCR accuracy. Compare a small hand-transcribed validation set before running all groups or replacing school-worker data.

In [ ]:
if not RUN_OK:
    print('No completed bulk run: downloads intentionally unavailable.')
else:
    EXPORT_OK = False
    try:
        if not RUN_OK:
            raise RuntimeError('Cell 3 did not finish successfully; no current downloads')
        helpers['validate_publish'](root, OUTPUT, ENV_ROOT, ARCHIVE)
        helpers['invalidate_ready'](OUTPUT)
        subprocess.run([*WORKER_COMMAND, 'export', str(root), str(OUTPUT),
                        str(ENV_ROOT), RUN_ID, '--archive', str(ARCHIVE)],
                       env=child_env, cwd=SCRATCH, check=True)
        download = helpers['checked_ready'](root, OUTPUT, ENV_ROOT, RUN_ID, ARCHIVE)
        from IPython.display import FileLink, display
        technical = ARCHIVE.with_name(ARCHIVE.stem + '_diagnostics.zip')
        receipt = helpers['read_json'](OUTPUT / 'technical-publication.json')
        if receipt['run_id'] != RUN_ID or receipt['sha256'] != helpers['file_digest'](technical):
            raise RuntimeError('Technical ZIP receipt mismatch')
        display(FileLink(str(download)))
        display(FileLink(str(technical)))
        EXPORT_OK = True
    except BaseException:
        EXPORT_OK = False
        if RUN_OK:
            helpers['invalidate_current_ready'](root, OUTPUT, ENV_ROOT, RUN_ID)
        print('WARNING: export failed; an older ocr.zip may remain, not a current result.', flush=True)
        raise
